<div dir="rtl">

# חיזוי עלות תביעות בביטוח רכב

המחברת הראשית של הפרויקט, לפי סדר סעיפי המטלה.

מחברות החקירה (cleaning_driver, daniel_prepare, target_eda) נמצאות ב-repo ולא משתנות.

## הכנה: ספריות וגרסאות

RANDOM_STATE מוגדר פעם אחת כאן, ומשמש בכל מקום שיש בו אקראיות. את הגרסאות שמודפסות נעתיק ל-requirements.txt, כי model.pkl נטען רק באותה גרסה של scikit-learn שבה הוא נשמר.

</div>

In [1]:
# ייבוא ספריות, קבוע אקראיות אחד לכל הפרויקט ובדיקת גרסאות
import sys
import numpy as np
import pandas as pd
import sklearn

RANDOM_STATE = 42

print("Python:", sys.version.split()[0])
print("pandas:", pd.__version__)
print("numpy:", np.__version__)
print("scikit-learn:", sklearn.__version__)

Python: 3.13.9
pandas: 2.2.3
numpy: 2.1.3
scikit-learn: 1.6.1


<div dir="rtl">

## טעינת הדאטה ומבט ראשון

הדאטה כולל 45,000 נהגים ו-30 עמודות. מציגים חמש שורות אקראיות ולא את הראשונות, כי הקובץ עלול להיות מסודר לפי משהו.

</div>

In [2]:
# טעינת הדאטה והצגת חמש שורות אקראיות

df = pd.read_csv("dataset.csv")
print(df.shape)
df.sample(5, random_state=RANDOM_STATE)

(45000, 30)


,customer_id_hash,agent_code,country,area_type,region_density,winter_severity,driver_age,gender,marital_status,license_years,...,annual_km_estimated,pct_intercity,pct_night_driving,parking_type,current_premium,claim_count,at_fault,days_to_settlement,police_report_filed,annual_claim_cost
37979,0ebf89f0,A020,FR,suburban,806,3.0,63,M,married,36,...,66500.0,10,45,garage,4270,0,NaN,NaN,NaN,0
9911,e7bedf11,A143,US,rural,24,0.0,38,M,divorced,20,...,17500.0,15,5,garage,1860,0,NaN,NaN,NaN,0
43386,9ff342a8,A060,FR,suburban,1115,3.8,84,M,married,66,...,3000.0,0,10,garage,1950,0,NaN,NaN,NaN,0
13822,b66c36b7,A004,DE,urban,10949,6.6,48,M,divorced,30,...,NaN,20,30,lot,13810,0,NaN,NaN,NaN,0
44810,30fbec1d,A004,US,rural,100,6.4,80,M,married,62,...,15500.0,55,10,garage,5160,0,NaN,NaN,NaN,0


<div dir="rtl">

בודקים איזה סוג נתונים pandas נתן לכל עמודה. עמודה שאמורה להיות מספרית ונקראה כטקסט מרמזת על ערכים מלוכלכים.

</div>

In [3]:
# סוג הנתונים בכל עמודה, כדי לזהות עמודות מספר שנקראו כטקסט
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45000 entries, 0 to 44999
Data columns (total 30 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   customer_id_hash       45000 non-null  object 
 1   agent_code             45000 non-null  object 
 2   country                45000 non-null  object 
 3   area_type              45000 non-null  object 
 4   region_density         45000 non-null  int64  
 5   winter_severity        45000 non-null  float64
 6   driver_age             45000 non-null  int64  
 7   gender                 45000 non-null  object 
 8   marital_status         45000 non-null  object 
 9   license_years          45000 non-null  int64  
 10  prior_claims_5y        41858 non-null  float64
 11  no_claims_years        45000 non-null  int64  
 12  violations_3y          45000 non-null  int64  
 13  vehicle_type           45000 non-null  object 
 14  vehicle_age            45000 non-null  int64  
 15  en

<div dir="rtl">

ערכים חסרים כפי שהם בקובץ. זו לא התמונה המלאה, כי חלק מהחוסרים מוסווים כמספרים, וכך נראה בתא הבא.

</div>

In [4]:
# כמות הערכים החסרים בכל עמודה, רק בעמודות שיש בהן חוסרים
missing = df.isna().sum()
missing[missing > 0]

prior_claims_5y         3142
annual_km_estimated     4311
at_fault               29207
days_to_settlement     29207
police_report_filed    29207
dtype: int64

<div dir="rtl">

סטטיסטיקה של העמודות המספריות. המינימום והמקסימום חושפים ערכים לא הגיוניים, שחוקי הניקוי יטפלו בהם בהמשך.

</div>

In [5]:
# סטטיסטיקה של עמודות המספר, כדי לזהות ערכים חריגים ולא הגיוניים
df.describe().T

,count,mean,std,min,25%,50%,75%,max
region_density,45000.0,1886.716311,2161.133202,4.0,498.0,1151.0,2593.0,20000.0
winter_severity,45000.0,3.459389,2.214855,0.0,1.7,3.3,5.0,10.0
driver_age,45000.0,48.010511,16.613355,18.0,35.0,47.0,60.0,90.0
license_years,45000.0,28.853867,16.866459,0.0,15.0,28.0,41.0,95.0
prior_claims_5y,41858.0,1.039706,1.384058,0.0,0.0,1.0,2.0,6.0
no_claims_years,45000.0,6.750933,8.101940,0.0,1.0,3.0,10.0,30.0
violations_3y,45000.0,0.416467,0.717039,0.0,0.0,0.0,1.0,6.0
vehicle_age,45000.0,7.638844,5.313188,0.0,4.0,6.0,10.0,25.0
engine_hp,45000.0,143.920800,66.350986,12.0,94.0,137.0,178.0,650.0
safety_features_count,45000.0,3.983378,2.028846,0.0,2.0,4.0,6.0,8.0


<div dir="rtl">

## 2.2 מניעת Data Leakage

המודל חוזה עלות תביעות של נהג חדש בתחילת השנה. מותר להשתמש רק בשדות שידועים לפני חתימת החוזה.

בשלב זה בודקים כל עמודה חשודה, ומוכיחים בנתונים אם היא מדליפה מידע מהעתיד.

</div>

<div dir="rtl">

### מספר התביעות: claim_count

אם מספר התביעות השנה גדול מאפס בדיוק כשיש עלות, העמודה חושפת את התשובה.

</div>

In [6]:
# בדיקה האם מספר התביעות חופף בדיוק לעלות התביעות
pd.crosstab(df["claim_count"] > 0, df["annual_claim_cost"] > 0,
            rownames=["claim_count > 0"], colnames=["cost > 0"])

cost > 0,False,True
claim_count > 0,,
False,29207,0
True,0,15793


<div dir="rtl">

### פרטי התאונה: at_fault, days_to_settlement, police_report_filed

פרטי התאונה קיימים רק כשהייתה תביעה. בודקים אם גם החוסר עצמו חושף את התשובה.

</div>

In [7]:
# בדיקה האם שלוש עמודות התאונה ריקות בדיוק כשאין תביעה
for col in ["at_fault", "days_to_settlement", "police_report_filed"]:
    print(col)
    print(pd.crosstab(df[col].isna(), df["annual_claim_cost"] > 0,
                      rownames=["missing"], colnames=["cost > 0"]), "\n")

at_fault
cost > 0  False  True 
missing               
False         0  15793
True      29207      0 

days_to_settlement
cost > 0  False  True 
missing               
False         0  15793
True      29207      0 

police_report_filed
cost > 0  False  True 
missing               
False         0  15793
True      29207      0 



<div dir="rtl">

### הפרמיה: current_premium

לפי מילון השדות זו הפרמיה ששולמה בסוף השנה. בודקים אם היא קשורה לתביעות של אותה שנה.

</div>

In [8]:
# בדיקת הקשר בין הפרמיה של סוף השנה לבין עלות התביעות
print("Correlation with target:", round(df["current_premium"].corr(df["annual_claim_cost"]), 3))
df.groupby(df["annual_claim_cost"] > 0)["current_premium"].mean()

Correlation with target: 0.577


annual_claim_cost
False    3188.265484
True     6214.693852
Name: current_premium, dtype: float64

<div dir="rtl">

### שנים ללא תביעה: no_claims_years

אם השדה היה מתעדכן בסוף השנה, נהגים שאשמים בתאונה השנה היו מופיעים כמעט כולם עם 0.

</div>

In [9]:
# בדיקה האם שנות הנהיגה ללא תביעה עודכנו בסוף השנה
df.groupby(df["at_fault"].fillna(-1))["no_claims_years"].apply(lambda s: (s == 0).mean())

at_fault
-1.0    0.146506
 0.0    0.264279
 1.0    0.347816
Name: no_claims_years, dtype: float64

<div dir="rtl">

### מזהים: customer_id_hash ו-agent_code

עמודות מנהליות. בודקים אם המזהה ייחודי וכמה סוכנים יש. את ההשפעה של הסוכן בדקנו בבדיקת ערבוב במחברת cleaning_driver.

</div>

In [10]:
# בדיקת העמודות המנהליות מזהה לקוח וקוד סוכן
print("customer_id_hash unique:", df["customer_id_hash"].is_unique)
print("agent_code values:", df["agent_code"].nunique())

customer_id_hash unique: True
agent_code values: 300


<div dir="rtl">

### מסקנות סעיף 2.2

עמודות Leakage שיוסרו: claim_count, at_fault, days_to_settlement, police_report_filed, current_premium. כולן נוצרות במהלך שנת הביטוח או בסופה, ולכן הן לא ידועות ביום החתימה.

עמודות שיוסרו כי אין להן ערך לחיזוי: customer_id_hash (מזהה ייחודי) ו-agent_code (בדיקת ערבוב במחברת cleaning_driver הראתה שהפיזור בין 300 הסוכנים, 807, לא גדול מהפיזור המקרי, 803).

עמודה שנבדקה ונמצאה תקינה: no_claims_years. רק 35% מהאשמים בתאונה השנה מופיעים עם 0, ולכן השדה לא עודכן בסוף השנה.

</div>

In [11]:
# רשימות העמודות שיוסרו מהמודל: דליפת מידע, ומזהים בלי ערך לחיזוי
LEAKAGE_COLS = ["claim_count", "at_fault", "days_to_settlement", "police_report_filed", "current_premium"]
ID_COLS = ["customer_id_hash", "agent_code"]